# Den fuktige kjelleren

## Termiske nettverk, fukttransport og kondensfare

### Pilotprosjekt for Matematikk 1, VVS og bygg

En kjellerbod ble innredet omkring 1982. En kald betongvegg mot terreng ble foret ut innvendig med mineralull, trestendere, plastfolie og gips. Veggen ser tørr og pen ut fra rommet, men kan ha et kaldt og fuktutsatt sjikt skjult bak isolasjonen.

Kjelleren består samtidig av tre luftsoner:

1. vaskerom med periodisk fuktkilde,
2. bod med den risikable ytterveggen,
3. trapperom mot første etasje.

Prosjektet undersøker hvordan veggoppbygning, luftutveksling, ventilasjon og fuktkilder påvirker temperaturen og fuktnivået ved det kritiske sjiktet.

Prosjektet har fem deler:

1. **Flerlagsvegg:** et tridiagonalt lineært system for temperaturnoder
2. **Tre fuktsoner:** et lineært system for stasjonær fuktfordeling
3. **Nettverksmatrisen:** oppbygging av sonekoblingen som $B^TQB$
4. **Dynamisk fukttransport:** en vektor-ODE løst med Euler
5. **Fuktmoder:** diagonalisering, tidskonstanter og fysisk tolkning

### Læringsmål

Etter prosjektet skal du kunne

- bygge en termisk nettverksmatrise fra lagmotstander,
- tolke hver matriserad som en energibalanse,
- kontrollere varmefluksen gjennom en lagdelt vegg,
- bygge et fuktnettverk fra luftmengder og forbindelser,
- løse stasjonære temperatur- og fuktproblemer som $Ax=b$,
- bruke absolutte fuktstørrelser i en massebalanse,
- beregne relativ fuktighet og kondensfare ved en kald flate,
- skrive fukttransporten som en vektor-ODE,
- bruke Euler på et tidsvarierende sonesystem,
- diagonaliserer en symmetrisk koblingsmatrise,
- tolke egenvektorer som karakteristiske fuktmønstre.

### Viktig avgrensning

Modellen kan påvise lave temperaturer og høy relativ fuktighet ved et skjult sjikt. Den beregner ikke kapillærsug, regnvannsinntrenging, materialfukt, muggvekst eller uttørking inne i konstruksjonen. Resultatene er derfor et pedagogisk risikosignal, ikke dokumentasjon av en fuktsikker veggløsning.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Referansekjelleren

## Luftsoner

Vi bruker følgende scenarioverdier:

- vaskerom: $45\ \mathrm{m^3}$,
- bod: $55\ \mathrm{m^3}$,
- trapperom: $35\ \mathrm{m^3}$.

Luftutveksling mellom sonene:

$$q_{12}=20\ \mathrm{m^3/h},\qquad q_{23}=30\ \mathrm{m^3/h}.$$

Ventilasjon mot ute:

$$q_{1u}=25,\qquad q_{2u}=8,\qquad q_{3u}=15\quad \mathrm{m^3/h}.$$

Stasjonære fuktkilder:

$$\dot m_1=80,\qquad \dot m_2=8,\qquad \dot m_3=0\quad \mathrm{g/h}.$$

Verdiene er valgt som et tydelig undervisningsscenario. De er ikke generelle prosjekteringsverdier.

In [ ]:
sone_navn = ["vaskerom", "bod", "trapperom"]
volum = np.array([45.0, 55.0, 35.0])  # m^3

q_ute_m3h = np.array([25.0, 8.0, 15.0])
q12_m3h = 20.0
q23_m3h = 30.0

fuktkilde_gh = np.array([80.0, 8.0, 0.0])

# SI-enheter
q_ute = q_ute_m3h/3600.0
q12 = q12_m3h/3600.0
q23 = q23_m3h/3600.0
fuktkilde = fuktkilde_gh/(1000.0*3600.0)

print("Ventilasjon i m^3/s:", q_ute)
print("Fuktkilder i kg/s:", fuktkilde)

# Del A: Temperatur gjennom 1982-veggen

## A.1 Referanseveggen

Fra jord mot rom modelleres veggen som

```text
jord
│
│ eldre utvendig fuktsikring, ikke modellert termisk
│
200 mm betong
│
20 mm stillestående luftspalte
│
75 mm mineralull mellom trestendere
│
plastfolie, neglisjerbar termisk motstand
│
13 mm gips
│
kjellerbod
```

Plastfolien får ikke en egen termisk motstand, men er viktig i modellkritikken fordi den kan redusere uttørking mot rommet.

Pilotdata:

| Lag | Tykkelse | Varmeledningsevne |
|---|---:|---:|
| Betong | $0.20$ m | $1.7$ W/(m K) |
| Luftspalte | oppgitt motstand | $R'=0.15$ m²K/W |
| Mineralull | $0.075$ m | $0.040$ W/(m K) |
| Gips | $0.013$ m | $0.25$ W/(m K) |

Vi bruker veggareal $A=12\ \mathrm{m^2}$, jordtemperatur $8^\circ$C og bodtemperatur $18^\circ$C.

In [ ]:
A_vegg = 12.0
T_jord = 8.0
T_bod = 18.0

lag_navn = ["betong", "luftspalte", "mineralull", "gips", "innvendig overflate"]

R_betong_areal = 0.20/1.7
R_luft_areal = 0.15
R_ull_areal = 0.075/0.040
R_gips_areal = 0.013/0.25
R_si_areal = 0.13

R_areal = np.array([
    R_betong_areal,
    R_luft_areal,
    R_ull_areal,
    R_gips_areal,
    R_si_areal
])

# Samlet termisk konduktans i W/K
G_lag = A_vegg/R_areal

for navn, R, G in zip(lag_navn, R_areal, G_lag):
    print(f"{navn:23s}: R'={R:7.4f} m^2K/W, G={G:8.3f} W/K")

## A.2 Termisk nettverksmatrise

Vi plasserer fire ukjente temperaturnoder mellom de fem termiske motstandene:

```text
T_jord --G1-- T1 --G2-- T2 --G3-- T3 --G4-- T4 --G5-- T_bod
```

$T_3$ ligger på kald side av den innvendige isolasjonen og brukes som kritisk skjult temperatur.

For en indre node gjelder at summen av varmestrømmene inn er null. Dette gir en symmetrisk, tridiagonal matrise.

## Oppgave A1: Bygg veggmatrisen

Bruk $G_1,\ldots,G_5$ og bygg systemet

$$A_TT=b_T.$$

Første rad skal representere balansen i $T_1$, og siste rad balansen i $T_4$.

In [ ]:
G1, G2, G3, G4, G5 = G_lag

A_T = np.array([
    [..., ..., 0.0, 0.0],
    [..., ..., ..., 0.0],
    [0.0, ..., ..., ...],
    [0.0, 0.0, ..., ...]
], dtype=float)

b_T = np.array([
    ...,
    0.0,
    0.0,
    ...
])

print("A_T =
", A_T)
print("b_T =", b_T)

## Oppgave A2: Løs temperaturprofilen

Løs systemet og skriv ut temperaturnodene. Kontroller residualet.

In [ ]:
T_noder = ...
print("Temperaturnoder:", T_noder)
print("Residualnorm:", ...)

T1, T2, T3_kritisk, T4 = T_noder
print("Kritisk skjult temperatur:", T3_kritisk, "grader C")

## Oppgave A3: Kontroller varmefluksen

Beregn varmeeffekten gjennom hvert ledd:

$$Q_1=G_1(T_{jord}-T_1),$$

$$Q_2=G_2(T_1-T_2),$$

og tilsvarende videre mot rommet.

Alle fem verdiene skal være omtrent like når samme fortegnsretning brukes.

In [ ]:
temperatur_med_rand = np.concatenate([[T_jord], T_noder, [T_bod]])
Q_lag = np.array([
    G_lag[j]*(temperatur_med_rand[j] - temperatur_med_rand[j+1])
    for j in range(5)
])

print("Varmeeffekt gjennom leddene:", Q_lag)
print("Største avvik fra middelverdien:", ...)

## A.3 Matriseegenskaper

Undersøk:

- symmetri,
- determinant,
- egenverdier,
- om matrisen er positiv definit.

Forklar hvorfor positive konduktanser gir en stabil og entydig stasjonær temperaturfordeling.

In [ ]:
print("Symmetrisk:", ...)
print("det(A_T) =", ...)
print("Egenverdier:", ...)

# Del B: Stasjonær fuktfordeling mellom tre soner

## B.1 Fuktstørrelser

Vi bruker vanndampkonsentrasjon

$$c=\frac{m_v}{V},$$

målt i $\mathrm{kg/m^3}$, som bevart størrelse.

Relativ fuktighet beregnes senere fra

$$
\boxed{\varphi=\frac{c}{c_{met}(T)}.}
$$

Metningsdamptrykket beregnes med den oppgitte tilnærmingen

$$
p_{met}(T)=610.94\exp\left(\frac{17.625T}{T+243.04}\right),
$$

og metningskonsentrasjonen med

$$c_{met}(T)=\frac{p_{met}(T)}{R_v(T+273.15)}.$$

In [ ]:
R_v = 461.5


def metningsdamptrykk(T_C):
    T_C = np.asarray(T_C, dtype=float)
    return 610.94*np.exp(17.625*T_C/(T_C + 243.04))


def metningskonsentrasjon(T_C):
    return metningsdamptrykk(T_C)/(R_v*(T_C + 273.15))


def konsentrasjon_fra_RH(T_C, RH):
    return RH*metningskonsentrasjon(T_C)


def relativ_fuktighet(T_C, c):
    return c/metningskonsentrasjon(T_C)

## B.2 Uteforhold

Bruk først et sommerpunkt:

$$T_u=23^\circ C,\qquad \varphi_u=65\%.$$

Beregn uteluftens vanndampkonsentrasjon. Sammenlign gjerne med vinterluft senere.

In [ ]:
T_ute_sommer = 23.0
RH_ute_sommer = 0.65
c_ute = ...
print("Uteluftens konsentrasjon:", c_ute, "kg/m^3")

## B.3 Sonebalanser

For luftutveksling mellom sonene brukes bidraget

$$q_{ij}(c_j-c_i).$$

Ved stasjonære forhold får vi

$$
\boxed{Kc=b.}
$$

Matrisen skal være

$$
K=
\begin{pmatrix}
q_{1u}+q_{12}&-q_{12}&0\\
-q_{12}&q_{2u}+q_{12}+q_{23}&-q_{23}\\
0&-q_{23}&q_{3u}+q_{23}
\end{pmatrix}.
$$

## Oppgave B1: Bygg og løs fuktmatrisen

Høyresiden er

$$
b=
\begin{pmatrix}
q_{1u}c_u+\dot m_1\\
q_{2u}c_u+\dot m_2\\
q_{3u}c_u+\dot m_3
\end{pmatrix}.
$$

In [ ]:
K_fukt = np.array([
    [..., ..., 0.0],
    [..., ..., ...],
    [0.0, ..., ...]
], dtype=float)

b_fukt = q_ute*c_ute + fuktkilde

c_sone = ...
print("K =
", K_fukt)
print("Konsentrasjoner:", c_sone)
print("Residualnorm:", ...)

## Oppgave B2: Relativ fuktighet i rommene

Bruk sone-temperaturene

$$T=(19,16,20)^\circ C.$$

Beregn relativ fuktighet i alle sonene.

In [ ]:
T_sone = np.array([19.0, 16.0, 20.0])
RH_sone = ...

for navn, c_i, rh_i in zip(sone_navn, c_sone, RH_sone):
    print(f"{navn:12s}: c={c_i:.6f} kg/m^3, RH={100*rh_i:.1f} %")

# Del C: Bygg sonekoblingen som $B^TQB$

## C.1 Forbindelsesmatrisen

Forbindelsene mellom sonene er

$$1\leftrightarrow2,\qquad2\leftrightarrow3.$$

Vi bruker

$$
B=
\begin{pmatrix}
1&-1&0\\
0&1&-1
\end{pmatrix}.
$$

Luftmengdene samles i

$$Q=\operatorname{diag}(q_{12},q_{23}).$$

Da er koblingsmatrisen mellom sonene

$$
\boxed{K_{mellom}=B^TQB.}
$$

Ventilasjonen mot ute legges til som en diagonalmatrise

$$K_u=\operatorname{diag}(q_{1u},q_{2u},q_{3u}).$$

## Oppgave C1: Konstruer nettverksmatrisen

Beregn $B^TQB+K_u$ og kontroller at dette gir den samme matrisen som i del B.

In [ ]:
B_kobling = np.array([
    [1.0, -1.0, 0.0],
    [0.0, 1.0, -1.0]
])

Q_luft = ...
K_mellom = ...
K_ute = ...
K_bygd = ...

print("K_mellom =
", K_mellom)
print("K_bygd =
", K_bygd)
print("Forskjell fra del B:", ...)

## C.2 Tolk matrisen

Forklar:

- hvorfor koblingene gir negative elementer utenfor diagonalen,
- hvorfor diagonalelementene er summer av luftmengder,
- hvorfor $K_{mellom}\mathbf 1=0$,
- hvorfor luftutveksling mellom rom fordeler fukt, men ikke fjerner fukt fra hele kjelleren.

In [ ]:
print("K_mellom @ 1 =", ...)
print("Egenverdier til K_mellom:", ...)
print("Egenverdier til K_fukt:", ...)

# Del D: Koble vegg og sonefukt

Fra del A har vi temperaturen $T_3$ ved det skjulte, kalde sjiktet bak innvendig isolasjon. Fra del B har vi vanndampkonsentrasjonen i boden, $c_2$.

Hvis bodluft lekker inn til sjiktet uten å miste vanndamp, blir den lokale relative fuktigheten

$$
\boxed{
\varphi_{kritisk}
=
\frac{c_{bod}}{c_{met}(T_3)}.}
$$

Verdier over 100 prosent betyr at modellen forutsier kondens. Høye verdier under 100 prosent kan fortsatt være ugunstige over tid, men prosjektet bruker ikke en egen muggmodell.

## Oppgave D1: Beregn kondensindikatoren

Sammenlign

- relativ fuktighet i bodluften,
- relativ fuktighet ved det skjulte sjiktet.

In [ ]:
c_bod = c_sone[1]
RH_bod = relativ_fuktighet(T_sone[1], c_bod)
RH_kritisk = ...

print("RH i bodluft:", 100*RH_bod, "%")
print("RH ved skjult sjikt:", 100*RH_kritisk, "%")
print("Kondens ifølge modellen:", RH_kritisk > 1.0)

## Oppgave D2: Tiltak virker på ulike deler av problemet

Undersøk minst tre av følgende:

- øk bodtemperaturen,
- øk avtrekket fra vaskerommet,
- øk eller reduser luftutvekslingen mellom vaskerom og bod,
- reduser fuktkilden i vaskerommet,
- bruk tørrere uteluft,
- endre veggoppbygningen,
- legg inn en enkel avfukter som negativ fuktkilde.

Rapporter både $T_3$, $c_{bod}$ og $\varphi_{kritisk}$.

# Del E: Dynamisk fuktfordeling

## E.1 Vektor-ODE

La

$$c(t)=(c_1,c_2,c_3)^T.$$

Volummatrisen er

$$
M_V=\operatorname{diag}(V_1,V_2,V_3).
$$

Massebalansen blir

$$
\boxed{
M_V\dot c=b(t)-K(t)c.}
$$

Ved hvert Euler-steg løser vi

$$M_V\dot c=b-Kc$$

med `np.linalg.solve`.

In [ ]:
M_V = np.diag(volum)


def uteforhold(t_timer):
    # Forenklet døgnvariasjon: varm og fuktig dag, kjøligere natt.
    if 8.0 <= (t_timer % 24.0) < 20.0:
        T_u, RH_u = 23.0, 0.65
    else:
        T_u, RH_u = 13.0, 0.75
    return T_u, RH_u, konsentrasjon_fra_RH(T_u, RH_u)


def dynamiske_kilder(t_timer):
    kilde = np.array([15.0, 8.0, 0.0])  # g/h grunnlast
    # Tørking av klær mellom kl. 18 og 22.
    if 18.0 <= (t_timer % 24.0) < 22.0:
        kilde[0] += 180.0
    return kilde/(1000.0*3600.0)


def fukt_ode(t_s, c):
    t_timer = t_s/3600.0
    _, _, c_u = uteforhold(t_timer)
    b = q_ute*c_u + dynamiske_kilder(t_timer)
    return ...

## Oppgave E1: Euler gjennom to døgn

Start i den stasjonære sommertilstanden fra del B. Bruk tidssteg på ett minutt, og kontroller deretter med et mindre steg.

In [ ]:
def euler_system(f, x0, sluttid, h):
    n = int(round(sluttid/h))
    t = np.linspace(0.0, n*h, n + 1)
    X = np.zeros((n + 1, len(x0)))
    X[0] = x0

    for k in range(n):
        X[k+1] = ...

    return t, X


t_E, c_E = euler_system(fukt_ode, c_sone, 48*3600.0, 60.0)

plt.figure(figsize=(11, 5))
for j, navn in enumerate(sone_navn):
    plt.plot(t_E/3600.0, 1000*c_E[:, j], label=navn)
plt.xlabel("Tid, timer")
plt.ylabel("Vanndampkonsentrasjon, g/m^3")
plt.grid(); plt.legend(); plt.show()

## Oppgave E2: Relativ fuktighet og kondensfare gjennom tiden

Bruk de faste sonetemperaturene i hovedmodellen. Beregn relativ fuktighet i hver sone og ved det kritiske veggsjiktet.

In [ ]:
RH_E = c_E/metningskonsentrasjon(T_sone)[None, :]
RH_kritisk_E = c_E[:, 1]/metningskonsentrasjon(T3_kritisk)

fig, ax = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
for j, navn in enumerate(sone_navn):
    ax[0].plot(t_E/3600.0, 100*RH_E[:, j], label=navn)
ax[0].set_ylabel("Relativ fuktighet, %")
ax[0].grid(); ax[0].legend()

ax[1].plot(t_E/3600.0, 100*RH_kritisk_E, label="skjult sjikt")
ax[1].axhline(100, color="black", linestyle="--", label="metning")
ax[1].set_xlabel("Tid, timer")
ax[1].set_ylabel("Lokal relativ fuktighet, %")
ax[1].grid(); ax[1].legend()
plt.show()

## E.2 Behovsstyrt avtrekk, valgfri utvidelse

En enkel styring kan øke avtrekket fra vaskerommet når rommet er fuktigere enn uteluften eller når relativ fuktighet overstiger en valgt grense.

Dette gir en stykkevis modell fordi $K(t)$ endres når avtrekket slås opp eller ned.

Sammenlign:

- kontinuerlig høyt avtrekk,
- kontinuerlig lavt avtrekk,
- behovsstyrt avtrekk,
- avfukter som negativ fuktkilde.

Vurder både fuktrisiko og unødvendig ventilasjon når uteluften er fuktig.

# Del F: Diagonalisering og fuktmoder

For å få en tydelig modalanalyse bruker vi først en symmetrisk forenkling:

- tre like sonevolumer $V$,
- lik ventilasjon $q$ mot ute,
- lik kobling $q_m$ mellom nabosoner.

Da blir

$$
K_{sym}=
\begin{pmatrix}
q+q_m&-q_m&0\\
-q_m&q+2q_m&-q_m\\
0&-q_m&q+q_m
\end{pmatrix}.
$$

Siden matrisen er symmetrisk, finnes en ortogonal diagonaliseringsmatrise:

$$
\boxed{K_{sym}=P\Lambda P^T.}
$$

In [ ]:
V_sym = 45.0
q_sym = 15.0/3600.0
q_m = 25.0/3600.0

K_sym = np.array([
    [q_sym+q_m, -q_m, 0.0],
    [-q_m, q_sym+2*q_m, -q_m],
    [0.0, -q_m, q_sym+q_m]
])

print("K_sym =
", K_sym)

## Oppgave F1: Finn fuktmodene

Finn egenverdier og ortonormale egenvektorer. Kontroller

$$P^TP=I$$

og

$$K_{sym}=P\Lambda P^T.$$

Tolk særlig en mode som er proporsjonal med $(1,1,1)^T$.

In [ ]:
egenverdier, P = ...
Lambda = ...

print("Egenverdier:", egenverdier)
print("Egenvektorer:
", P)
print("P^T P =
", ...)
print("P Lambda P^T =
", ...)

## F.2 Variabelbytte og tidskonstanter

Sett

$$c-c_u\mathbf 1=Pz.$$

Uten tidsvarierende pådrag får vi

$$
\boxed{V\dot z=-\Lambda z.}
$$

Modene er frakoblet:

$$V\dot z_i=-\lambda_i z_i.$$

Tidskonstanten til mode $i$ er

$$
\boxed{\tau_i=\frac{V}{\lambda_i}.}
$$

## Oppgave F2: Tolk modene

1. Beregn tidskonstantene i timer.
2. Identifiser fellesmoden.
3. Identifiser en mode som beskriver forskjell mellom yttersonene.
4. Identifiser en mode der midtsonen avviker fra yttersonene.
5. Forklar hvorfor økt luftutveksling mellom rommene reduserer forskjellsmodene, men ikke nødvendigvis tørker hele kjelleren raskere.

In [ ]:
tau_s = ...
tau_timer = tau_s/3600.0
print("Tidskonstanter i timer:", tau_timer)

## Oppgave F3: Direkte og modal simulering

Velg en starttilstand der bare vaskerommet er fuktigere enn de andre. Simuler både

- det direkte systemet,
- de tre frakoblede modalligningene.

Transformer tilbake med

$$c=c_u\mathbf1+Pz$$

og sammenlign løsningene.

In [ ]:
# Implementer direkte og modal simulering og sammenlign maksimal forskjell.

# Fordypning: Sammenlign tre vegger

## Vegg 1: 1982-risikoveggen

- 200 mm betong
- luftspalte
- 75 mm innvendig mineralull i trestendere
- plastfolie
- gips

## Vegg 2: Uinnredet kjellervegg

- 200 mm betong
- mineralsk puss mot rom

Denne har større varmetap og kaldere synlig innvendig overflate, men ingen skjult isolert trevegg med organiske materialer.

## Vegg 3: Utbedret hovedprinsipp

- forbedret utvendig fuktsikring
- 100 mm utvendig isolasjon
- 200 mm betong
- mineralsk innvendig overflate

Studentene skal sammenligne temperaturprofil og kondensindikator, men skal ikke konkludere med at en vegg er fuktsikker utelukkende på grunnlag av denne modellen.

## Fordypningsoppgave: Foreslå en utbedring

Endre maksimalt tre forhold i referansemodellen. Mulige tiltak:

- fjern innvendig trestendervegg,
- flytt isolasjonen til utsiden,
- forbedre utvendig fuktsikring og drenering som kvalitativt tiltak,
- øk bodtemperaturen,
- reduser fuktkilden,
- endre ventilasjonsstrategien,
- legg inn avfukter,
- bruk en mineralsk og uttørkingsvennlig innvendig overflate.

Rapporter før og etter:

- varmefluks,
- synlig innvendig overflatetemperatur,
- kritisk skjult temperatur,
- fuktkonsentrasjon i alle soner,
- relativ fuktighet i boden,
- relativ fuktighet ved kritisk sjikt,
- modellens begrensninger.

Materialdata som gruppen velger selv, skal kildehenvises og enhetene dokumenteres.

# Modellkritikk

Diskuter minst seks punkter:

- Jordsiden representeres med én fast temperatur.
- Fuktgjennomgang fra jord og mur beregnes ikke fysisk.
- Luftspalten representeres med én fast termisk motstand.
- Plastfolien har nesten ingen termisk betydning, men stor fuktteknisk betydning.
- Hver luftsones temperatur og fuktinnhold antas uniforme.
- Luftmengdene antas kjente og konstante i hovedmodellen.
- Relativ fuktighet ved skjult sjikt beregnes som om bodluften når sjiktet uten endring i vanndampinnhold.
- Materialfukt, sorpsjon og uttørking er utelatt.
- Kapillærsug, regnvann og sviktende drenering er utelatt.
- Modellen har ingen egen mugg- eller råtemodell.
- Kondens fjernes ikke fra luftbalansen i hovedmodellen.
- Sommer- og nattluften er svært forenklet.
- Euler krever kontroll av tidssteget.
- Symmetrimodellen i del F er en analysemodell, ikke den nøyaktige referansekjelleren.

## Mulige videreføringer

- dynamiske temperaturer i vegglagene,
- hygroskopisk lagring i treverk og papp,
- kondens som fuktsluk og latent varme,
- trykkstyrt infiltrasjon,
- værdata og grunnforhold,
- fukttilskudd fra dusj og tørketrommel,
- avfukter med hysterese,
- målte temperatur- og RH-data,
- kobling til en mer detaljert varme- og fuktmodell i senere emner.

# Oppsummering

Skriv en kort rapport der du forklarer

1. hvordan vegglagene ga en tridiagonal temperaturmatrise,
2. hvorfor hver matriserad er en energibalanse,
3. hvordan varmefluksen ble kontrollert,
4. hvordan luftforbindelsene ga en fuktmatrise,
5. hvordan $B^TQB$ bygde koblingen mellom sonene,
6. hvorfor absolutt fuktinnhold brukes i massebalansen,
7. hvordan veggtemperatur og sonefukt ble koblet til en kondensindikator,
8. hvordan det stasjonære systemet ble til en vektor-ODE,
9. hva felles- og forskjellsmodene betyr fysisk,
10. hvorfor temperatur- og luftfuktighetsmodellen ikke alene kan dokumentere en fuktsikker kjellervegg.

## Faglig bakgrunn

Prosjektet er inspirert av kjente risikokonstruksjoner i eldre innredede kjellere: kald mur mot terreng, innvendig isolasjon, treverk og tette sjikt med begrenset uttørking. Eldre kjellere kan dessuten ha mangelfull utvendig fuktsikring og være uegnet for oppholdsrom uten omfattende tiltak.

Studentene trenger ikke eksterne kilder for å gjennomføre hovedløpet. Selvvalgte materialdata i fordypningen skal dokumenteres.